# Lab 2: N-Gram Language Models from Scratch 

**Objective:** In this lab, you will build Bigram and Trigram language models from scratch using Python. You will learn how to extract n-grams, calculate probabilities, apply Laplace (Add-1) smoothing to handle unseen words, and evaluate your model using Perplexity.


### Outline:
1. **Task 1:** Text Preprocessing & Padding
2. **Task 2:** Extracting N-grams
3. **Task 3:** Building the Raw Bigram/Trigram Model
4. **Task 4:** Implementing Laplace (Add-1) Smoothing
5. **Task 5:** Calculating Perplexity on Held-out Text

In [4]:
import math
from collections import defaultdict, Counter

train_text = [
    "India is a beautiful country in South Asia",
    "The capital of India is New Delhi",
    "India is famous for its rich culture and diverse heritage",
    "Many vibrant festivals are celebrated in India throughout the year"
]

test_text = [
    "New Delhi is the capital of India",
    "India has a rich culture and beautiful festivals"
]

## Task 1: Text Preprocessing & Padding (Easy)

Language models need to know where a sentence starts and ends. We do this by adding special padding tokens. 
* For a **Bigram** model, we need one start token `<s>` and one end token `</s>`.
* For a **Trigram** model, we need two start tokens `<s> <s>` so the first word has a two-word context.

**Your Task:** Write a function that takes a sentence, lowercases it, splits it into words, and adds the appropriate number of start and end tokens based on `n`.

In [5]:
def preprocess_sentence(sentence, n):

    tokens = sentence.lower().split()
    
    start_tokens = ['<s>'] * (n - 1)
    end_tokens = ['</s>']
    
    return start_tokens + tokens + end_tokens

print("Bigram padding:", preprocess_sentence(train_text[0], n=2))
print("Trigram padding:", preprocess_sentence(train_text[0], n=3))

Bigram padding: ['<s>', 'india', 'is', 'a', 'beautiful', 'country', 'in', 'south', 'asia', '</s>']
Trigram padding: ['<s>', '<s>', 'india', 'is', 'a', 'beautiful', 'country', 'in', 'south', 'asia', '</s>']


## Task 2: Extracting N-Grams (Easy)

Now that we can pad a sentence, we need to extract the n-grams (sequences of n words).

**Your Task:** Write a function that takes a list of padded tokens and returns a list of n-gram tuples. For example, given `['<s>', 'india', 'is']`, a bigram extraction should return `[('<s>', 'india'), ('india', 'is')]`.

In [6]:
def get_ngrams(tokens, n):
  
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        ngrams.append(ngram)
    return ngrams

padded_tokens = preprocess_sentence(train_text[0], n=3)
print("Tokens:", padded_tokens)
get_ngrams(padded_tokens, n=3)

Tokens: ['<s>', '<s>', 'india', 'is', 'a', 'beautiful', 'country', 'in', 'south', 'asia', '</s>']


[('<s>', '<s>', 'india'),
 ('<s>', 'india', 'is'),
 ('india', 'is', 'a'),
 ('is', 'a', 'beautiful'),
 ('a', 'beautiful', 'country'),
 ('beautiful', 'country', 'in'),
 ('country', 'in', 'south'),
 ('in', 'south', 'asia'),
 ('south', 'asia', '</s>')]

## Task 3: Building the Raw N-gram Model (Medium)

To calculate the probability of a word given its context, we need to count how many times the n-gram occurs and divide it by how many times the context occurs. 

For a Trigram model:
$$P(w_i | w_{i-2}, w_{i-1}) = \frac{count(w_{i-2}, w_{i-1}, w_i)}{count(w_{i-2}, w_{i-1})}$$

**Your Task:** Write a class or set of functions to count the frequencies of all contexts (n-1 grams) and all full n-grams in the training corpus. Determine the total vocabulary size (V).

In [17]:
class NGramModel:
    def __init__(self, n):
        self.n = n
        self.ngram_counts = defaultdict(int)
        self.context_counts = defaultdict(int)
        self.vocabulary = set()
        self.vocab_size = 0
        
    def train(self, corpus):
        for sentence in corpus:
            tokens = preprocess_sentence(sentence, self.n)
            
            # Add to vocabulary (excluding start tokens)
            for token in tokens:
                if token != '<s>':
                    self.vocabulary.add(token)
            
            # Extract n-grams
            ngrams = get_ngrams(tokens, self.n)
            
            # Count n-grams and contexts
            for ngram in ngrams:
                self.ngram_counts[ngram] += 1
                
                # Context is everything except the last word in the n-gram
                context = ngram[:-1]
                self.context_counts[context] += 1
                
        self.vocab_size = len(self.vocabulary)
        print(f"Model trained! n={self.n}, Vocab size={self.vocab_size}")

# Test the training
bigram_model = NGramModel(n=2)
bigram_model.train(train_text)
print("Count of ('india', 'is'):", bigram_model.ngram_counts[('india', 'is')])
print("Count of context ('india',):", bigram_model.context_counts[('india',)])

Model trained! n=2, Vocab size=29
Count of ('india', 'is'): 3
Count of context ('india',): 4


In [15]:
bigram_model.context_counts

defaultdict(int,
            {('<s>',): 4,
             ('india',): 4,
             ('is',): 3,
             ('a',): 1,
             ('beautiful',): 1,
             ('country',): 1,
             ('in',): 2,
             ('south',): 1,
             ('asia',): 1,
             ('the',): 2,
             ('capital',): 1,
             ('of',): 1,
             ('new',): 1,
             ('delhi',): 1,
             ('famous',): 1,
             ('for',): 1,
             ('its',): 1,
             ('rich',): 1,
             ('culture',): 1,
             ('and',): 1,
             ('diverse',): 1,
             ('heritage',): 1,
             ('many',): 1,
             ('vibrant',): 1,
             ('festivals',): 1,
             ('are',): 1,
             ('celebrated',): 1,
             ('throughout',): 1,
             ('year',): 1})

## Task 4: Laplace (Add-1) Smoothing (Medium)

If we encounter an n-gram in our test set that never appeared in our training set, its raw count will be 0. This gives a probability of 0, which breaks our model and makes perplexity infinite.

We fix this using Laplace (Add-1) Smoothing:
$$P_{Laplace}(w_i | context) = \frac{count(context, w_i) + 1}{count(context) + V}$$
where V is the total vocabulary size.

**Your Task:** Implement a method inside our model to calculate this smoothed probability.

In [8]:
def get_smoothed_prob(model, ngram):
    """
    Calculates the Laplace smoothed probability of an n-gram.
    ngram: tuple, e.g., ('india', 'is')
    """
    context = ngram[:-1]
    
    # Apply Add-1 smoothing formula
    numerator = model.ngram_counts[ngram] + 1
    denominator = model.context_counts[context] + model.vocab_size
    
    return numerator / denominator

# Test smoothing
print("Smoothed Prob of ('india', 'is'):", get_smoothed_prob(bigram_model, ('india', 'is')))
print("Smoothed Prob of ('india', 'monsoons'):", get_smoothed_prob(bigram_model, ('india', 'monsoons'))) # OOV test

Smoothed Prob of ('india', 'is'): 0.12121212121212122
Smoothed Prob of ('india', 'monsoons'): 0.030303030303030304


## Task 5: Computing Perplexity (Medium)

Perplexity is the standard evaluation metric for language models. A lower perplexity indicates a better model that is less "surprised" by the test data.

The formula for Perplexity over a test sequence of N words is:
$$PP(W) = \sqrt[N]{\prod_{i=1}^N \frac{1}{P(w_i | context)}}$$

To avoid arithmetic underflow (multiplying many tiny fractions), we use log probabilities:
$$PP(W) = 2^{-\frac{1}{N} \sum \log_2 P(w_i | context)}$$

**Your Task:** Write a function to calculate the perplexity of a given test corpus using your smoothed probabilities.

In [9]:
def calculate_perplexity(model, test_corpus):
    total_log_prob = 0.0
    total_words = 0
    
    for sentence in test_corpus:
        tokens = preprocess_sentence(sentence, model.n)
        ngrams = get_ngrams(tokens, model.n)
        
        for ngram in ngrams:
            prob = get_smoothed_prob(model, ngram)
            # Use log base 2
            total_log_prob += math.log2(prob)
            
        # We count all words predicted (which equals the number of ngrams)
        total_words += len(ngrams)
        
    # Calculate average log probability and exponentiate
    avg_log_prob = total_log_prob / total_words
    perplexity = math.pow(2, -avg_log_prob)
    
    return perplexity

# Evaluate models
trigram_model = NGramModel(n=3)
trigram_model.train(train_text)

print(f"Bigram Model Perplexity on Test Data: {calculate_perplexity(bigram_model, test_text):.2f}")
print(f"Trigram Model Perplexity on Test Data: {calculate_perplexity(trigram_model, test_text):.2f}")

Model trained! n=3, Vocab size=29
Bigram Model Perplexity on Test Data: 22.60
Trigram Model Perplexity on Test Data: 24.81
